## **Revision 04 -- Main model with and without the creatinine missingness indicator**

**Purpose.** The response to reviewers (R2-7, R2-20) states that the main model includes an indicator of creatinine
missingness and that results are reported **with and without it**. Creatinine is present in 19.5% of deliveries without
SMM and in 62.9% of those with SMM, so the indicator carries information about *who was tested*, not only about the
value. This notebook measures how much of the main-model performance depends on that indicator.

**Design.** The same eligible delivery hospitalizations, the same outer partitions (same seed, patient-grouped, 5 folds x 3
repeats), the same inner tuning, stacking and recalibration as `02_grouped_validation_smm.ipynb`. Two analyses:

| Analysis | Predictors |
|---|---|
| `main` | age, hemoglobin, creatinine, **creatinine missingness indicator**, five categorical admission variables (replicates notebook 02) |
| `main_noind` | the same **without** the indicator (creatinine is median-imputed, so a missing value is not flagged) |

Because both analyses use the same splits and the same bootstrap samples of patients, the differences are **paired**.
Replicating `main` here (instead of reusing notebook 02) is what makes the paired comparison possible; its point
estimates should match `outputs_main.txt` up to numerical noise.

**Data-use rules.** Only aggregate results are printed; the only file written is `outputs_no_indicator.txt`
(aggregate text, listed in `.gitignore` through `outputs_*.txt`). Nothing patient-level is shown or saved.

**Run order.** Run all cells top to bottom with `QUICK = False` (about twice the time of the main run of notebook 02).
Set `QUICK = True` first for a smoke test.

### **1. Libraries, connection and settings**

In [1]:
import os, sys, time, math, warnings
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# notebooks/revision/ -> project root is two levels up
sys.path.append(os.path.abspath(os.path.join("..", "..")))
import libraries.database as db_lib
engine = db_lib.get_connect_database()

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

# ----------------------------------------------------------------------------- settings
QUICK        = False        # True = smoke test (small forests, 1 repeat, B = 50). Never report QUICK numbers.
SEED         = 20260101
OUTER_FOLDS  = 5
N_REPEATS    = 3            # repeats of the outer grouped CV (development instability)
INNER_FOLDS  = 3
BOOT         = 1000         # cluster bootstrap replicates for the main analysis
BOOT_SENS    = 300          # replicates for the sensitivity analyses
N_JOBS       = -1
N_TREES      = 300
XGB_TREES    = 200
APPLY_LIMITS = True         # set values outside the plausibility limits of the dictionary to missing
CAT_COLS     = ["ethnicity", "insurance", "language", "admission_type", "admission_location"]
DCA_T        = [0.005, 0.01, 0.02, 0.03, 0.05, 0.10, 0.15, 0.20]
EXPECTED     = {"eligible_deliveries": 19383, "smm": 445, "deaths": 3}   # from the audit counts (Q-F / Q-G)

if QUICK:
    OUTER_FOLDS, N_REPEATS, BOOT, BOOT_SENS = 3, 1, 50, 30
    N_TREES, XGB_TREES = 60, 50
print(f"QUICK={QUICK}  outer={OUTER_FOLDS}x{N_REPEATS}  inner={INNER_FOLDS}  B={BOOT}  trees={N_TREES}")

QUICK=False  outer=5x3  inner=3  B=1000  trees=300


### **2. Load the predictor table and the final dictionary**

`predictors_24h_v2` has one row per obstetric hospitalization with the predictors observed in the window
(admission - 24 h, admission + 24 h). `predictor_dictionary_final_v2` assigns every variable its role
(`main model`, `ICU sensitivity only`, `laboratory sensitivity ...`) and its plausibility limits.
Only counts are printed.

In [2]:
df = pd.read_sql("SELECT * FROM mimiciv.predictors_24h_v2", engine)
dic = pd.read_sql("""SELECT variable, model_role_final, missingness_indicator_main, plausible_low, plausible_high,
                            n_eligible_delivery, coverage_pct
                     FROM mimiciv.predictor_dictionary_final_v2""", engine)

for c in ["eligible_24h", "icu_in_24h", "is_delivery_v2", "smm_any", "died_same_admission"]:
    df[c] = df[c].fillna(False).astype(bool)
df["admittime"] = pd.to_datetime(df["admittime"])

VAR_MAIN = dic.loc[dic.model_role_final == "main model", "variable"].tolist()
VAR_ICU  = dic.loc[dic.model_role_final == "ICU sensitivity only", "variable"].tolist()
VAR_LAB  = dic.loc[dic.model_role_final.str.startswith("laboratory sensitivity"), "variable"].tolist()
IND_MAIN = dic.loc[dic.missingness_indicator_main.fillna(False).astype(bool), "variable"].tolist()

print("Dictionary roles ->", {"main": VAR_MAIN, "ICU only": len(VAR_ICU), "lab sensitivity": len(VAR_LAB),
                              "missingness indicator in main model": IND_MAIN})

# Plausibility limits (proposals pending clinical confirmation): out-of-range values become missing
if APPLY_LIMITS:
    n_set = {}
    for _, r in dic.dropna(subset=["plausible_low", "plausible_high"]).iterrows():
        v = r["variable"]
        if v in df.columns:
            bad = (df[v] < r["plausible_low"]) | (df[v] > r["plausible_high"])
            n_set[v] = int(bad.sum())
            df.loc[bad, v] = np.nan
    print("Values set to missing by plausibility limits (count per variable):", n_set)

# Missingness indicators (computed row by row from the raw value: no information crosses patients)
for v in set(IND_MAIN + VAR_LAB + VAR_ICU):
    df[f"{v}_missing"] = df[v].isna().astype(int)
df["icu_in_24h_bin"] = df["icu_in_24h"].astype(int)
df["y"] = df["smm_any"].astype(int)

# Audit checks against the counts already certified (aggregate only)
elig = df[df.eligible_24h & df.is_delivery_v2]
chk = {"eligible_deliveries": len(elig), "smm": int(elig.smm_any.sum()), "deaths": int(elig.died_same_admission.sum())}
for k, v in chk.items():
    print(f"{k}: {v}  (expected {EXPECTED[k]})  [{'OK' if v == EXPECTED[k] else 'CHECK: differs from the audit counts'}]")
print("Eligible any obstetric admissions:", int(df.eligible_24h.sum()),
      "| SMM:", int(df[df.eligible_24h].smm_any.sum()), "| deaths:", int(df[df.eligible_24h].died_same_admission.sum()))

Dictionary roles -> {'main': ['hemoglobin', 'creatinine'], 'ICU only': 14, 'lab sensitivity': 11, 'missingness indicator in main model': ['creatinine']}
Values set to missing by plausibility limits (count per variable): {'hemoglobin': 0, 'hematocrit': 0, 'creatinine': 1}
eligible_deliveries: 19383  (expected 19383)  [OK]
smm: 445  (expected 445)  [OK]
deaths: 3  (expected 3)  [OK]
Eligible any obstetric admissions: 23625 | SMM: 814 | deaths: 5


### **3. Analysis datasets and feature sets**

Each analysis is a (population, feature set, label) specification:

| Name | Population | Features |
|---|---|---|
| `main` | Eligible delivery hospitalizations | Main-model variables of the dictionary (hemoglobin, creatinine), creatinine missingness indicator, age, categorical admission variables |
| `first_admission` | Index (first) delivery hospitalization of each patient, if eligible | as `main` |
| `any_obstetric` | Eligible obstetric hospitalizations (delivery or not) | as `main` |
| `labs_extended` | as `main` | `main` + hematocrit and the other laboratory variables, each with a missingness indicator |
| `icu_vars` | as `main` | `main` + the ICU-only variables (each with a missingness indicator) + `icu_in_24h` |
| `label_ge2` | as `main` | as `main`, outcome = two or more SMM indicators (`smm_ge2`) |
| `no_icu_24h` | Eligible deliveries **without** an ICU admission in the first 24 h | as `main` |
| `icd10_only` | Eligible deliveries whose diagnoses are all ICD-10-CM (no ICD-9-CM diagnosis; decision D4) | as `main` |

In [3]:
def make_spec(name):
    d = df[df.eligible_24h].copy()
    num = ["age_at_admission"] + VAR_MAIN
    binc = [f"{v}_missing" for v in IND_MAIN]
    label = "y"
    if name in ("main", "main_noind", "labs_extended", "icu_vars", "label_ge2", "no_icu_24h", "icd10_only"):
        d = d[d.is_delivery_v2]
        if name == "no_icu_24h":
            d = d[~d.icu_in_24h]
    elif name == "first_admission":
        first = df[df.is_delivery_v2].sort_values(["subject_id", "admittime", "hadm_id"]).drop_duplicates("subject_id")
        d = d[d.hadm_id.isin(first.hadm_id)]
    elif name == "any_obstetric":
        pass
    else:
        raise ValueError(name)
    if name == "icd10_only":
        ver = pd.read_sql("SELECT hadm_id, has_icd9_dx, has_icd10_dx FROM mimiciv.cohort_obstetric_v2", engine)
        ver["has_icd9_dx"] = ver["has_icd9_dx"].fillna(False).astype(bool)
        ver["has_icd10_dx"] = ver["has_icd10_dx"].fillna(False).astype(bool)
        keep = ver.loc[ver.has_icd10_dx & ~ver.has_icd9_dx, "hadm_id"]
        d = d[d.hadm_id.isin(keep)]
    if name == "main_noind":
        binc = []                       # no missingness indicator for creatinine
    if name == "labs_extended":
        num = num + VAR_LAB
        binc = binc + [f"{v}_missing" for v in VAR_LAB if f"{v}_missing" not in binc]
    if name == "icu_vars":
        num = num + VAR_ICU
        binc = binc + [f"{v}_missing" for v in VAR_ICU] + ["icu_in_24h_bin"]
    if name == "label_ge2":
        g2 = pd.read_sql("SELECT hadm_id, smm_ge2 FROM mimiciv.cohort_obstetric_v2", engine)
        g2["smm_ge2"] = g2["smm_ge2"].fillna(False).astype(bool).astype(int)
        d = d.merge(g2, on="hadm_id", how="left")
        d["y_ge2"] = d["smm_ge2"].fillna(0).astype(int)
        label = "y_ge2"
    # drop columns that do not exist or never vary (a constant indicator carries no information)
    num  = [c for c in dict.fromkeys(num) if c in d.columns]
    binc = [c for c in dict.fromkeys(binc) if c in d.columns and d[c].nunique() > 1]
    cat  = [c for c in CAT_COLS if c in d.columns]
    d = d.reset_index(drop=True)
    return dict(name=name, data=d, num=num, bin=binc, cat=cat, label=label)

for nm in ["main", "main_noind"]:
    sp = make_spec(nm)
    d = sp["data"]
    print(f"{nm:16s} admissions={len(d):6d}  patients={d.subject_id.nunique():6d}  events={int(d[sp['label']].sum()):4d}  "
          f"numeric={len(sp['num'])}  binary={len(sp['bin'])}  categorical={len(sp['cat'])}")

main             admissions= 19383  patients= 13710  events= 445  numeric=3  binary=1  categorical=5
main_noind       admissions= 19383  patients= 13710  events= 445  numeric=3  binary=0  categorical=5


### **4. Core functions (preprocessing, learners, nested tuning, stacking, calibration)**

Everything that learns from data is inside the training portion of each split:

* `Winsor` clips each numeric column to the 0.5th-99.5th percentiles *of the training data*.
* The median imputer, the scaler and the one-hot encoder are fitted inside a `Pipeline`, once per fit.
* Hyperparameters are chosen by **inner patient-grouped CV** (average precision of the pooled out-of-fold predictions).
* The meta-learner of the ensemble is trained on the **out-of-fold** predictions of the base learners (never on in-sample predictions).
* The Platt calibrator is trained on out-of-fold predictions of the training patients; the outer test patients are only scored.

In [4]:
def logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

def expit(z):
    return 1.0 / (1.0 + np.exp(-z))

class Winsor(BaseEstimator, TransformerMixin):
    """Clip each column to training percentiles (fit on the training data only)."""
    def __init__(self, lo=0.5, hi=99.5):
        self.lo = lo; self.hi = hi
    def fit(self, X, y=None):
        X = np.asarray(X, float)
        self.lo_ = np.nanpercentile(X, self.lo, axis=0)
        self.hi_ = np.nanpercentile(X, self.hi, axis=0)
        return self
    def transform(self, X):
        return np.clip(np.asarray(X, float), self.lo_, self.hi_)

def make_prep(num, binc, cat):
    parts = [("num", Pipeline([("w", Winsor()),
                               ("imp", SimpleImputer(strategy="median", keep_empty_features=True)),
                               ("sc", StandardScaler())]), num)]
    if binc:
        parts.append(("bin", "passthrough", binc))
    if cat:
        parts.append(("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                                       ("oh", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                                            min_frequency=0.01, sparse_output=False))]), cat))
    return ColumnTransformer(parts)

def prep_frame(d, num, binc, cat):
    X = d[num + binc + cat].copy()
    for c in cat:
        X[c] = X[c].astype(object).where(X[c].notna(), np.nan)
    return X

# Small prespecified grids (class weights are options; no resampling)
GRID = {
    # Widened after the first run, in which several learners chose the edge of the original grid
    # (ET depth 8 of {4, 8}; RF depth 4 of {4, 8}; XGB depth 2 of {2, 4}; no class weight ever chosen).
    "LR":  [dict(C=C, cw=cw) for C in (0.01, 0.1, 1.0) for cw in (None, "balanced")],
    "RF":  [dict(depth=dp, cw=cw) for dp in (2, 4, 8) for cw in (None, "balanced_subsample")],
    "ET":  [dict(depth=dp, cw=cw) for dp in (4, 8, 12) for cw in (None, "balanced_subsample")],
    "XGB": [dict(depth=dp, spw=sp) for dp in (1, 2, 4) for sp in (1, "sqrt")],
}
MODELS = ["LR", "RF", "ET", "XGB", "Stack", "Stack_raw"]
BOOT_MODELS = ["LR", "RF", "ET", "XGB", "Stack"]
PAIRS = [("RF", "LR"), ("ET", "LR"), ("XGB", "LR"), ("Stack", "LR"), ("Stack", "XGB")]

def cfg_str(c):
    return ",".join(f"{k}={v}" for k, v in c.items())

def make_clf(name, cfg, ytr, seed):
    if name == "LR":
        return LogisticRegression(C=cfg["C"], class_weight=cfg["cw"], max_iter=3000)
    if name == "RF":
        return RandomForestClassifier(n_estimators=N_TREES, max_depth=cfg["depth"], min_samples_leaf=20,
                                      class_weight=cfg["cw"], n_jobs=N_JOBS, random_state=seed)
    if name == "ET":
        return ExtraTreesClassifier(n_estimators=N_TREES, max_depth=cfg["depth"], min_samples_leaf=20,
                                    class_weight=cfg["cw"], n_jobs=N_JOBS, random_state=seed)
    if name == "XGB":
        pos = max(int(ytr.sum()), 1); neg = max(len(ytr) - pos, 1)
        spw = math.sqrt(neg / pos) if cfg["spw"] == "sqrt" else 1.0
        return XGBClassifier(n_estimators=XGB_TREES, learning_rate=0.05, max_depth=cfg["depth"], subsample=0.8,
                             colsample_bytree=0.8, min_child_weight=5, scale_pos_weight=spw, eval_metric="logloss",
                             tree_method="hist", n_jobs=N_JOBS if N_JOBS and N_JOBS > 0 else 0, random_state=seed)
    raise ValueError(name)

def fit_predict(name, cfg, prep, Xtr, ytr, Xte, seed):
    pipe = Pipeline([("prep", clone(prep)), ("clf", make_clf(name, cfg, ytr, seed))])
    pipe.fit(Xtr, ytr)
    return pipe.predict_proba(Xte)[:, 1]

def platt(o, t, ytr):
    c = LogisticRegression(penalty=None, max_iter=1000).fit(logit(o).reshape(-1, 1), ytr)
    return c.predict_proba(logit(t).reshape(-1, 1))[:, 1]

def tune_stack_calibrate(Xtr, ytr, gtr, Xte, prep, seed):
    """Inside one outer training set: tune, build out-of-fold predictions, stack, calibrate; score the outer test set."""
    inner = list(StratifiedGroupKFold(INNER_FOLDS, shuffle=True, random_state=seed).split(Xtr, ytr, gtr))
    names = list(GRID)
    chosen, oof, pred_te = {}, {}, {}
    for name in names:
        best = None
        for cfg in GRID[name]:
            o = np.zeros(len(ytr))
            for tr, va in inner:
                o[va] = fit_predict(name, cfg, prep, Xtr.iloc[tr], ytr[tr], Xtr.iloc[va], seed)
            ap = average_precision_score(ytr, o)
            if best is None or ap > best[0]:
                best = (ap, cfg, o)
        chosen[name] = best[1]; oof[name] = best[2]
        pred_te[name] = fit_predict(name, best[1], prep, Xtr, ytr, Xte, seed)
    # explicit stacking: logistic meta-learner on out-of-fold base predictions (C = 0.1 as in the original framework)
    Z_oof = np.column_stack([logit(oof[n]) for n in names])
    Z_te = np.column_stack([logit(pred_te[n]) for n in names])
    meta = LogisticRegression(C=0.1, max_iter=2000).fit(Z_oof, ytr)
    stack_te_raw = meta.predict_proba(Z_te)[:, 1]
    # cross-fitted meta out-of-fold predictions -> calibrator never sees the predictions of its own training rows
    mo = np.zeros(len(ytr))
    for tr, va in StratifiedGroupKFold(INNER_FOLDS, shuffle=True, random_state=seed + 1).split(Z_oof, ytr, gtr):
        m = LogisticRegression(C=0.1, max_iter=2000).fit(Z_oof[tr], ytr[tr])
        mo[va] = m.predict_proba(Z_oof[va])[:, 1]
    out = {n: platt(oof[n], pred_te[n], ytr) for n in names}
    out["Stack"] = platt(mo, stack_te_raw, ytr)
    out["Stack_raw"] = stack_te_raw
    return out, chosen

def run_cv(spec, label=""):
    d = spec["data"]
    X = prep_frame(d, spec["num"], spec["bin"], spec["cat"])
    y = d[spec["label"]].values.astype(int)
    g = d["subject_id"].values
    prep = make_prep(spec["num"], spec["bin"], spec["cat"])
    n = len(y)
    preds = {m: np.full((N_REPEATS, n), np.nan) for m in MODELS}
    chosen_log = []
    split_log = []
    t0 = time.time(); done = 0; total = N_REPEATS * OUTER_FOLDS
    for r in range(N_REPEATS):
        outer = StratifiedGroupKFold(OUTER_FOLDS, shuffle=True, random_state=SEED + r)
        for k, (tr, te) in enumerate(outer.split(X, y, g)):
            assert len(set(g[tr]) & set(g[te])) == 0, "patient present in training and test"
            split_log.append(dict(repeat=r + 1, fold=k + 1,
                                  train_adm=len(tr), train_pat=len(set(g[tr])), train_events=int(y[tr].sum()),
                                  train_event_pat=len(set(g[tr][y[tr] == 1])),
                                  test_adm=len(te), test_pat=len(set(g[te])), test_events=int(y[te].sum()),
                                  test_event_pat=len(set(g[te][y[te] == 1])), shared_patients=0))
            out, chosen = tune_stack_calibrate(X.iloc[tr], y[tr], g[tr], X.iloc[te], prep, SEED + 100 * r + k)
            for m in MODELS:
                preds[m][r, te] = out[m]
            for nme, cfg in chosen.items():
                chosen_log.append((nme, cfg_str(cfg)))
            done += 1
            if done == 1:
                el = time.time() - t0
                print(f"[{spec['name']}] first outer fold took {el/60:.1f} min -> projected total about {el*total/60:.0f} min")
    print(f"[{spec['name']}] cross-validation finished in {(time.time()-t0)/60:.1f} min; "
          f"every admission scored exactly once per repeat: {bool(np.isfinite(preds['Stack']).all())}")
    return dict(spec=spec, y=y, g=g, preds=preds, chosen=chosen_log, X=X, prep=prep, splits=pd.DataFrame(split_log))

### **5. Metrics, cluster bootstrap and paired differences**

All metrics are computed on the pooled out-of-fold predictions of each repeat and then averaged over repeats.
The bootstrap resamples **patients** (all admissions of a patient move together) and recomputes every metric with
patient-count weights, so intervals reflect the repeated-admission structure. Differences against logistic
regression use the **same** bootstrap samples (paired).

In [5]:
def citl(y, z, w):
    """Calibration-in-the-large: intercept with the slope fixed at 1 (weighted Newton)."""
    a = 0.0
    for _ in range(40):
        pr = expit(z + a)
        g = np.sum(w * (y - pr)); h = np.sum(w * pr * (1 - pr))
        if h < 1e-12:
            break
        step = g / h; a += step
        if abs(step) < 1e-9:
            break
    return a

def calib_slope(y, z, w):
    """Calibration slope (weighted logistic recalibration, Newton)."""
    a, b = 0.0, 1.0
    for _ in range(40):
        pr = expit(a + b * z)
        r = w * (y - pr); v = w * pr * (1 - pr)
        gr = np.array([r.sum(), (r * z).sum()])
        H = np.array([[v.sum(), (v * z).sum()], [(v * z).sum(), (v * z * z).sum()]])
        try:
            dlt = np.linalg.solve(H, gr)
        except np.linalg.LinAlgError:
            break
        a += dlt[0]; b += dlt[1]
        if np.abs(dlt).max() < 1e-9:
            break
    return b

def top_threshold(p, w, k):
    order = np.argsort(-p)
    cw = np.cumsum(w[order])
    idx = int(np.searchsorted(cw, k * cw[-1]))
    return p[order][min(idx, len(p) - 1)]

def metrics_one(y, p, w):
    z = logit(p)
    m = {}
    m["auroc"] = roc_auc_score(y, p, sample_weight=w)
    m["auprc"] = average_precision_score(y, p, sample_weight=w)
    m["brier"] = float(np.average((p - y) ** 2, weights=w))
    m["slope"] = calib_slope(y, z, w)
    m["citl"] = citl(y, z, w)
    m["oe"] = float(np.sum(w * y) / np.sum(w * p))
    edges = np.quantile(p, np.linspace(0, 1, 11)[1:-1])
    bins = np.searchsorted(edges, p)
    W = w.sum(); ece = 0.0
    for b in range(10):
        mk = bins == b; wb = w[mk].sum()
        if wb > 0:
            ece += wb / W * abs(np.sum(w[mk] * y[mk]) / wb - np.sum(w[mk] * p[mk]) / wb)
    m["ece"] = ece
    ev = np.sum(w * y)
    for k in (0.05, 0.10):
        sel = p >= top_threshold(p, w, k)
        m[f"sens_top{int(k*100)}"] = np.sum(w[sel] * y[sel]) / ev
        m[f"ppv_top{int(k*100)}"] = np.sum(w[sel] * y[sel]) / np.sum(w[sel])
    for pt in DCA_T:
        sel = p >= pt
        tp = np.sum(w[sel] * y[sel]); fp = np.sum(w[sel] * (1 - y[sel]))
        m[f"nb_{pt}"] = tp / W - fp / W * pt / (1 - pt)
    return m

def all_metrics(y, preds, w, models, strata):
    """Metrics for each model, averaged over repeats. Returns dict[(model, metric)] and the per-repeat list."""
    per = []
    for r in range(N_REPEATS):
        cur = {}
        for mname in models:
            for k, v in metrics_one(y, preds[mname][r], w).items():
                cur[(mname, k)] = v
        prev = np.sum(w * y) / w.sum()
        for pt in DCA_T:
            cur[("TreatAll", f"nb_{pt}")] = prev - (1 - prev) * pt / (1 - pt)
        for sname, mask in strata.items():
            for mname in ("LR", "Stack"):
                if mname not in models:
                    continue
                wm = w[mask]; ym = y[mask]; pm = preds[mname][r][mask]
                if np.sum(wm * ym) >= 1 and np.sum(wm * (1 - ym)) >= 1:
                    cur[(mname, f"auroc_{sname}")] = roc_auc_score(ym, pm, sample_weight=wm)
                    cur[(mname, f"auprc_{sname}")] = average_precision_score(ym, pm, sample_weight=wm)
                else:
                    cur[(mname, f"auroc_{sname}")] = np.nan
                    cur[(mname, f"auprc_{sname}")] = np.nan
        per.append(cur)
    keys = per[0].keys()
    mean = {k: float(np.nanmean([c[k] for c in per])) if not all(np.isnan(c[k]) for c in per) else np.nan for k in keys}
    return mean, per

def bootstrap(res, B, strata, seed):
    y, g, preds = res["y"], res["g"], res["preds"]
    ug, inv = np.unique(g, return_inverse=True)
    G = len(ug)
    rng = np.random.default_rng(seed)
    out = {}
    t0 = time.time()
    for b in range(B):
        cnt = np.bincount(rng.integers(0, G, G), minlength=G)
        w = cnt[inv].astype(float)
        if np.sum(w * y) < 1:
            continue
        mean, _ = all_metrics(y.astype(float), preds, w, BOOT_MODELS, strata)
        for k, v in mean.items():
            out.setdefault(k, []).append(v)
        for a_, b_ in PAIRS:            # paired differences (same bootstrap sample)
            for met in ("auroc", "auprc", "brier") + tuple(f"nb_{pt}" for pt in DCA_T):
                out.setdefault((f"{a_}-{b_}", met), []).append(mean[(a_, met)] - mean[(b_, met)])
        if b == 4:
            el = time.time() - t0
            print(f"  bootstrap: 5 replicates took {el:.0f} s -> projected {el*B/5/60:.1f} min for B={B}")
    return {k: np.array(v) for k, v in out.items()}

### **6. Reporting function**

Prints aggregate tables only: discrimination, calibration, paired differences against logistic regression, the share of
events captured in the top 5 % / 10 % of predicted risk, net benefit, development instability across repeats and the
hyperparameters chosen in the inner loops.

In [6]:
def _ci(boot, key, d=3):
    a = boot.get(key)
    if a is None or len(a) < 20:
        return ""
    lo, hi = np.nanpercentile(a, [2.5, 97.5])
    return f" ({lo:.{d}f}-{hi:.{d}f})"

def decile_table(res, models=("LR", "Stack")):
    y = res["y"].astype(float)
    out = None
    for m in models:
        acc = []
        for r in range(N_REPEATS):
            p = res["preds"][m][r]
            edges = np.quantile(p, np.linspace(0, 1, 11)[1:-1])
            b = np.searchsorted(edges, p)
            acc.append(pd.DataFrame({"b": b, "p": p, "y": y}).groupby("b").agg(n=("y", "size"), pred=("p", "mean"), obs=("y", "mean")))
        mean = sum(acc) / len(acc)
        t = pd.DataFrame({"decile": mean.index + 1, "n": mean["n"].round(0).astype(int)}) if out is None else out
        t[f"{m}_pred"] = mean["pred"].round(4).values
        t[f"{m}_obs"] = mean["obs"].round(4).values
        out = t
    return out

def plot_calibration(res):
    import matplotlib.pyplot as plt
    t = decile_table(res)
    fig, ax = plt.subplots(figsize=(5, 5))
    mx = float(max(t["LR_pred"].max(), t["LR_obs"].max(), t["Stack_pred"].max(), t["Stack_obs"].max())) * 1.1
    ax.plot([0, mx], [0, mx], color="grey", lw=1, ls="--", label="ideal")
    ax.plot(t["LR_pred"], t["LR_obs"], "o-", label="Logistic regression")
    ax.plot(t["Stack_pred"], t["Stack_obs"], "s-", label="Stacked ensemble")
    ax.set_xlabel("Mean predicted risk (decile)"); ax.set_ylabel("Observed SMM rate")
    ax.set_title(f"Calibration by decile -- {res['spec']['name']}"); ax.legend(); plt.show()   # displayed only, not saved

def emit(name, fn):
    """Print the aggregate report and also keep it as a text file (aggregates only) so it can be shared without truncation."""
    import io, contextlib
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        fn()
    text = buf.getvalue()
    print(text)
    path = f"outputs_{name}.txt"
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(text)
    print(f"[aggregate report saved as notebooks/revision/{path}  ({len(text)} characters)]")

def run_analysis(name, B):
    spec = make_spec(name)
    res = run_cv(spec)
    d = spec["data"]
    strata = {}
    if name in ("main", "main_noind"):
        strata = {"icu1": d["icu_in_24h"].values.astype(bool), "icu0": ~d["icu_in_24h"].values.astype(bool)}
    y = res["y"].astype(float)
    res["point"], res["per"] = all_metrics(y, res["preds"], np.ones(len(y)), MODELS + [], strata)
    res["boot"] = bootstrap(res, B, strata, SEED + 7)
    res["strata"] = strata
    return res

def report(res):
    sp = res["spec"]; d = sp["data"]; y = res["y"]
    pt, per, bt = res["point"], res["per"], res["boot"]
    n, ev, npat = len(y), int(y.sum()), d.subject_id.nunique()
    print("=" * 110)
    print(f"ANALYSIS {sp['name']}: {n} admissions, {npat} patients, {ev} events ({100*ev/n:.2f}%), "
          f"numeric={len(sp['num'])}, binary={len(sp['bin'])}, categorical={len(sp['cat'])}")
    print(f"Repeats={N_REPEATS}, outer folds={OUTER_FOLDS}, inner folds={INNER_FOLDS}, bootstrap B={len(next(iter(bt.values()))) if bt else 0}, QUICK={QUICK}")
    print("=" * 110)

    sl = res["splits"]
    print("\n[0] Patients per outer partition (repeat 1 shown; every split verified to share no patient between training and test)")
    print(sl[sl.repeat == 1].drop(columns=["repeat"]).to_string(index=False))
    print(f"    Across all {len(sl)} splits: test patients per fold {sl.test_pat.min()}-{sl.test_pat.max()}, "
          f"event-bearing test patients per fold {sl.test_event_pat.min()}-{sl.test_event_pat.max()}, "
          f"patients present on both sides: {int(sl.shared_patients.sum())}")

    rows = []
    for m in MODELS:
        r = {"model": m}
        for met, dd in [("auroc", 3), ("auprc", 3), ("brier", 5)]:
            r[met] = f"{pt[(m, met)]:.{dd}f}" + (_ci(bt, (m, met), dd) if m in BOOT_MODELS else "")
        r["sd_auroc_repeats"] = f"{np.std([c[(m,'auroc')] for c in per], ddof=1):.4f}" if len(per) > 1 else "n/a"
        r["sd_auprc_repeats"] = f"{np.std([c[(m,'auprc')] for c in per], ddof=1):.4f}" if len(per) > 1 else "n/a"
        rows.append(r)
    print("\n[1] Discrimination and overall accuracy (mean over repeats; 95% cluster-bootstrap CI)")
    print(f"    prevalence = {y.mean():.4f}  (AUPRC of a non-informative model equals the prevalence)")
    print(pd.DataFrame(rows).to_string(index=False))

    rows = []
    for m in MODELS:
        r = {"model": m}
        for met, dd in [("slope", 2), ("citl", 2), ("oe", 2), ("ece", 4)]:
            r[met] = f"{pt[(m, met)]:.{dd}f}" + (_ci(bt, (m, met), dd) if m in BOOT_MODELS else "")
        rows.append(r)
    print("\n[2] Calibration (slope ideal 1; calibration-in-the-large ideal 0; O/E ideal 1; ECE with 10 quantile bins)")
    print(pd.DataFrame(rows).to_string(index=False))

    rows = []
    for a_, b_ in PAIRS:
        r = {"comparison (first minus second)": f"{a_} - {b_}"}
        for met, dd in [("auroc", 3), ("auprc", 3), ("brier", 5)]:
            r[met] = f"{pt[(a_, met)] - pt[(b_, met)]:+.{dd}f}" + _ci(bt, (f"{a_}-{b_}", met), dd)
        rows.append(r)
    print("\n[3] Paired differences (same bootstrap samples); the last row compares the ensemble with XGBoost")
    print(pd.DataFrame(rows).to_string(index=False))

    rows = []
    for m in BOOT_MODELS:
        rows.append({"model": m,
                     "sens_top5%": f"{pt[(m,'sens_top5')]:.3f}" + _ci(bt, (m, "sens_top5")),
                     "ppv_top5%": f"{pt[(m,'ppv_top5')]:.3f}",
                     "sens_top10%": f"{pt[(m,'sens_top10')]:.3f}" + _ci(bt, (m, "sens_top10")),
                     "ppv_top10%": f"{pt[(m,'ppv_top10')]:.3f}"})
    print("\n[4] Events captured among the 5% / 10% of admissions with the highest predicted risk")
    print(pd.DataFrame(rows).to_string(index=False))

    rows = []
    for m in BOOT_MODELS + ["TreatAll"]:
        rows.append({"strategy": m, **{f"pt={t:g}": round(pt[(m, f'nb_{t}')], 5) for t in DCA_T}})
    rows.append({"strategy": "TreatNone", **{f"pt={t:g}": 0.0 for t in DCA_T}})
    print("\n[5] Net benefit (decision curve) at the stated threshold probabilities")
    print(pd.DataFrame(rows).to_string(index=False))
    rows = []
    for a_, b_ in [("Stack", "LR"), ("RF", "LR"), ("ET", "LR"), ("XGB", "LR"), ("Stack", "XGB")]:
        rows.append({"comparison": f"{a_} - {b_}", **{f"pt={t:g}": f"{pt[(a_,f'nb_{t}')]-pt[(b_,f'nb_{t}')]:+.5f}" + _ci(bt, (f"{a_}-{b_}", f"nb_{t}"), 5) for t in DCA_T[:6]}})
    print("    Paired net-benefit differences:")
    print(pd.DataFrame(rows).to_string(index=False))

    if res["strata"]:
        rows = []
        for sname, lab in [("icu1", "ICU admission within 24 h"), ("icu0", "no ICU within 24 h")]:
            mk = res["strata"][sname]
            r = {"stratum": lab, "n": int(mk.sum()), "events": int(y[mk].sum())}
            for m in ("LR", "Stack"):
                r[f"{m}_auroc"] = (f"{pt[(m, 'auroc_' + sname)]:.3f}" + _ci(bt, (m, "auroc_" + sname))) if not np.isnan(pt[(m, 'auroc_' + sname)]) else "n/a"
                r[f"{m}_auprc"] = (f"{pt[(m, 'auprc_' + sname)]:.3f}") if not np.isnan(pt[(m, 'auprc_' + sname)]) else "n/a"
            rows.append(r)
        print("\n[6] Performance by ICU admission within the first 24 h (same predictions, stratified evaluation)")
        print(pd.DataFrame(rows).to_string(index=False))

    print("\n[8] Calibration by decile of predicted risk (outer-test predictions; mean over repeats)")
    print(decile_table(res).to_string(index=False))

    ch = pd.DataFrame(res["chosen"], columns=["learner", "config"])
    tab = ch.value_counts().rename("outer_fits").reset_index().sort_values(["learner", "outer_fits"], ascending=[True, False])
    print(f"\n[7] Hyperparameters selected in the inner loops (out of {N_REPEATS*OUTER_FOLDS} outer fits per learner)")
    print(tab.to_string(index=False))

### **7. Run both analyses (nested, patient-grouped)**

`main` replicates the main analysis of notebook 02; `main_noind` removes the creatinine missingness indicator. The
bootstrap inside `run_analysis` uses `BOOT_SENS` replicates; the paired comparison in the next section uses its own
bootstrap (`BOOT`).

In [7]:
res_ind = run_analysis("main", BOOT_SENS)
res_no  = run_analysis("main_noind", BOOT_SENS)
sp_i, sp_n = res_ind["spec"], res_no["spec"]
print("predictors with indicator   :", sp_i["num"], sp_i["bin"], sp_i["cat"])
print("predictors without indicator:", sp_n["num"], sp_n["bin"], sp_n["cat"])
assert np.array_equal(res_ind["y"], res_no["y"]) and np.array_equal(res_ind["g"], res_no["g"]), "analyses must share rows and patients"

[main] first outer fold took 1.1 min -> projected total about 16 min
[main] cross-validation finished in 12.8 min; every admission scored exactly once per repeat: True
  bootstrap: 5 replicates took 2 s -> projected 2.5 min for B=300
[main_noind] first outer fold took 1.1 min -> projected total about 17 min
[main_noind] cross-validation finished in 14.8 min; every admission scored exactly once per repeat: True
  bootstrap: 5 replicates took 1 s -> projected 1.4 min for B=300
predictors with indicator   : ['age_at_admission', 'hemoglobin', 'creatinine'] ['creatinine_missing'] ['ethnicity', 'insurance', 'language', 'admission_type', 'admission_location']
predictors without indicator: ['age_at_admission', 'hemoglobin', 'creatinine'] [] ['ethnicity', 'insurance', 'language', 'admission_type', 'admission_location']


### **8. Paired comparison and report**

For each model the table gives the metric with the indicator, without it, and the difference (without minus with) with a
95% patient-level cluster-bootstrap interval from the same resamples of patients. A negative difference means the model
loses performance when the indicator is removed.

In [8]:
def light(y, p, w):
    z = logit(p)
    return {"auroc": roc_auc_score(y, p, sample_weight=w), "auprc": average_precision_score(y, p, sample_weight=w),
            "brier": float(np.average((p - y) ** 2, weights=w)), "slope": calib_slope(y, z, w),
            "oe": float(np.sum(w * y) / np.sum(w * p))}

def paired_indicator(res_a, res_b, B, seed):
    """res_a = with indicator, res_b = without indicator. Returns point estimates, bootstrap draws of both and of the differences."""
    y = res_a["y"].astype(float); g = res_a["g"]
    ug, inv = np.unique(g, return_inverse=True)
    rng = np.random.default_rng(seed)
    models = ["LR", "RF", "ET", "XGB", "Stack"]
    mets = ("auroc", "auprc", "brier", "slope", "oe")
    def one(w):
        out = {}
        for tag, res in (("with", res_a), ("without", res_b)):
            for m in models:
                acc = [light(y, res["preds"][m][r], w) for r in range(N_REPEATS)]
                for k in mets:
                    out[(m, tag, k)] = float(np.mean([a[k] for a in acc]))
        return out
    point = one(np.ones(len(y)))
    boot = {}
    t0 = time.time()
    for b in range(B):
        w = np.bincount(rng.integers(0, len(ug), len(ug)), minlength=len(ug))[inv].astype(float)
        if np.sum(w * y) < 1:
            continue
        cur = one(w)
        for k, v in cur.items():
            boot.setdefault(k, []).append(v)
        for m in models:
            for k in mets:
                boot.setdefault((m, "diff", k), []).append(cur[(m, "without", k)] - cur[(m, "with", k)])
        if b == 4:
            el = time.time() - t0
            print(f"  bootstrap: 5 replicates took {el:.0f} s -> projected {el*B/5/60:.1f} min for B={B}")
    return point, {k: np.array(v) for k, v in boot.items()}, models

def indicator_report(res_a, res_b, point, boot, models):
    ci = lambda key, d=3: (lambda lo, hi: f" ({lo:.{d}f}-{hi:.{d}f})")(*np.nanpercentile(boot[key], [2.5, 97.5]))
    y = res_a["y"]; d = res_a["spec"]["data"]
    nb = len(next(iter(boot.values())))
    print("=" * 110)
    print(f"CREATININE MISSINGNESS INDICATOR: SMM, eligible deliveries, n={len(y)}, patients={d.subject_id.nunique()}, "
          f"events={int(y.sum())}, prevalence={y.mean():.4f}")
    print(f"outer {OUTER_FOLDS}x{N_REPEATS}, inner {INNER_FOLDS}, paired bootstrap B={nb}, QUICK={QUICK}")
    print("with indicator   : numeric", res_a["spec"]["num"], "binary", res_a["spec"]["bin"], "categorical", len(res_a["spec"]["cat"]))
    print("without indicator: numeric", res_b["spec"]["num"], "binary", res_b["spec"]["bin"], "categorical", len(res_b["spec"]["cat"]))
    print("=" * 110)

    rows = []
    for m in models:
        r = {"model": m}
        for met, dd in (("auroc", 3), ("auprc", 3), ("brier", 5)):
            r[f"{met} with"] = f"{point[(m,'with',met)]:.{dd}f}" + ci((m, "with", met), dd)
            r[f"{met} without"] = f"{point[(m,'without',met)]:.{dd}f}" + ci((m, "without", met), dd)
            r[f"{met} diff (without-with)"] = f"{point[(m,'without',met)] - point[(m,'with',met)]:+.{dd}f}" + ci((m, "diff", met), dd)
        rows.append(r)
    print("\n[1] Discrimination and Brier score with and without the indicator (paired; 95% cluster-bootstrap intervals)")
    print(pd.DataFrame(rows).to_string(index=False))

    rows = []
    for m in models:
        rows.append({"model": m,
                     "slope with": f"{point[(m,'with','slope')]:.2f}", "slope without": f"{point[(m,'without','slope')]:.2f}",
                     "O/E with": f"{point[(m,'with','oe')]:.2f}", "O/E without": f"{point[(m,'without','oe')]:.2f}"})
    print("\n[2] Calibration after recalibration (slope; observed/expected)")
    print(pd.DataFrame(rows).to_string(index=False))

    rows = []
    for tag, res in (("with indicator", res_a), ("without indicator", res_b)):
        pt = res["point"]
        for sname, lab in (("icu1", "ICU admission within 24 h"), ("icu0", "no ICU within 24 h")):
            r = {"analysis": tag, "stratum": lab, "n": int(res["strata"][sname].sum()), "events": int(y[res["strata"][sname]].sum())}
            for m in ("LR", "Stack"):
                a = pt.get((m, f"auroc_{sname}"), np.nan); b = pt.get((m, f"auprc_{sname}"), np.nan)
                r[f"{m} auroc"] = "n/a" if np.isnan(a) else f"{a:.3f}"
                r[f"{m} auprc"] = "n/a" if np.isnan(b) else f"{b:.3f}"
            rows.append(r)
    print("\n[3] Performance by ICU admission within the first 24 h (point estimates, same predictions)")
    print(pd.DataFrame(rows).to_string(index=False))

    print("\n[4] Creatinine availability that the indicator encodes (eligible deliveries)")
    cr = d["creatinine"].notna()
    print(f"    creatinine present: {100*cr.mean():.1f}% overall | {100*cr[y==0].mean():.1f}% without SMM | {100*cr[y==1].mean():.1f}% with SMM")
    print(f"    ICU within 24 h: {100*cr[d.icu_in_24h.values].mean():.1f}% | no ICU within 24 h: {100*cr[~d.icu_in_24h.values].mean():.1f}%")

    for res, tag in ((res_a, "with indicator"), (res_b, "without indicator")):
        ch = pd.DataFrame(res["chosen"], columns=["learner", "config"])
        tab = ch.value_counts().rename("outer_fits").reset_index().sort_values(["learner", "outer_fits"], ascending=[True, False])
        print(f"\n[5] Hyperparameters selected in the inner loops, {tag} (out of {N_REPEATS*OUTER_FOLDS} outer fits per learner)")
        print(tab.to_string(index=False))

point_i, boot_i, models_i = paired_indicator(res_ind, res_no, BOOT, SEED + 21)
emit("no_indicator", lambda: indicator_report(res_ind, res_no, point_i, boot_i, models_i))

  bootstrap: 5 replicates took 1 s -> projected 3.6 min for B=1000
CREATININE MISSINGNESS INDICATOR: SMM, eligible deliveries, n=19383, patients=13710, events=445, prevalence=0.0230
outer 5x3, inner 3, paired bootstrap B=1000, QUICK=False
with indicator   : numeric ['age_at_admission', 'hemoglobin', 'creatinine'] binary ['creatinine_missing'] categorical 5
without indicator: numeric ['age_at_admission', 'hemoglobin', 'creatinine'] binary [] categorical 5

[1] Discrimination and Brier score with and without the indicator (paired; 95% cluster-bootstrap intervals)
model          auroc with       auroc without auroc diff (without-with)          auprc with       auprc without auprc diff (without-with)                brier with             brier without   brier diff (without-with)
   LR 0.780 (0.754-0.802) 0.695 (0.663-0.725)    -0.085 (-0.109--0.059) 0.154 (0.125-0.195) 0.139 (0.109-0.180)    -0.016 (-0.027--0.004) 0.02087 (0.01900-0.02266) 0.02115 (0.01922-0.02305)  +0.00027 (0.00008-0.000